# WordPiece Tokenization for NER Task

In [ ]:
import sys
from pathlib import Path
import json 
from transformers import AutoModelForTokenClassification, AutoTokenizer

PROJECT_ROOT = Path("/Users/robertagarcia/Desktop/learning/bert_symptom_ner")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# ========================
# LOAD TOKENIZER AND MODEL
# ========================
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

# ========================
# LOAD THE WORD TOKENIZED DATASET
# ========================
dataset = []
with open(f'{PROJECT_ROOT}/V03/data/synthetic_data_tokenized.jsonl', 'r') as f:
    for line in f:
        row = json.loads(line)
        dataset.append(json.loads(line))



In [ ]:
print("Dataset sample:")
dataset[0]

# Quickly Check how the Tokenizer Works

In [ ]:
# Load all data from JSONL into a list of dictionaries
ex1 = dataset[0]
print(ex1.keys())
print("Tokenize text")
toks = tokenizer(ex1['text'])
print(toks)

print("\nTokenize tokens")

print("Demonstrating is_split_into_words=False vs is_split_into_words=True:")
print("- Setting is_split_into_words=False assumes your input is a single string or a flat list of tokens as a string, so passing a list of word tokens (especially if nested) may not work as intended.")
print("- Setting is_split_into_words=True tells the tokenizer that the input is a list of words (not a string), so it will tokenize and align each word separately. This is typically required when you want to map predictions back to words in sequence labeling tasks. (If unsure: when working with a list of word tokens, True is likely what you want.)")

print("\nwith is_split_into_words=False (this will not work due to the nested lists)")
toks_ids = tokenizer(ex1['word_tokens'], is_split_into_words=False)
print(toks_ids)
# Will not work because of nested lists
# print("Tokens:\n", tokenizer.convert_ids_to_tokens(toks_ids['input_ids']))

print("\nwith is_split_into_words=True")
toks_ids = tokenizer(ex1['word_tokens'], is_split_into_words=True)
print(toks_ids)
print("Word ids:\n", toks_ids.word_ids())
print("Tokens:\n", tokenizer.convert_ids_to_tokens(toks_ids['input_ids']))

# [CLS] -> Classification token (start of sequence)
# [SEP] -> Separator token (end of sequence)

# Tokenization flow: words -> word ids -> token ids

# Tokenization

-> Assign -100 to special tokens:  [CLS], [SEP], pytorch will ignore predictions of these tokens


In [ ]:
# Run an example of how the tokenizer workds
ex = dataset[10]
print("Sample text: ", ex['text'])
word_tokens = ex['word_tokens']
print("Word tokens: ", word_tokens)
print("\n-------- TOKENIZE ! --------")
# TOKENIZE!
toks_ids = tokenizer(ex['word_tokens'],is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(toks_ids['input_ids'])
print("\nTokenized words (is_split_into_words=True):\n\t", tokens)
print("Each token id:\n\t", toks_ids['input_ids'])
word_ids = toks_ids.word_ids()
print("Mappings of tokens to word index in the list of word_tokens:\n\t", word_ids)


In [ ]:
# ===========================================================================
# Create wordpiece-tokenized dataset 
# ===========================================================================

# Prepare list for storing tokenized samples (optional, for downstream use)
tokenized_samples = []

for row in dataset:

    # Tokenize word tokens for each sample
    words = row['word_tokens']
    toks_ids = tokenizer(words, is_split_into_words=True)
    input_ids = toks_ids['input_ids']
    word_ids = toks_ids.word_ids()

    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    token_labels = []
    previous_word_id = None

    for i, tok in enumerate(input_ids):
        word_id = word_ids[i]

        # Special tokens
        if word_id is None:
            token_labels.append("None") # -100
            print(f"word_id: {word_id}, token_label: None, original_label: None, fixed_label: None")
        # First subword of a word
        elif word_id != previous_word_id:
            token_label = row['word_labels'][word_id]
            token_labels.append(token_label)
            print(f"word_id: {word_id}, token_label: {token_label}, original_label: {token_label}, fixed_label: {token_label}")
        # Continuation subwords
        else:
            original_label = row['word_labels'][word_id]
            # If it's a B- tag, convert it to I-
            if isinstance(original_label, str) and original_label.startswith("B-"):
                fixed_label = original_label.replace("B-", "I-")
            else:
                fixed_label = original_label
            token_labels.append(fixed_label)
            print(f"word_id: {word_id}, token_label: None, original_label: {original_label}, fixed_label: {fixed_label}")
        previous_word_id = word_id

    # Prepare record for saving
    tokenized_sample = {
        "text": row["text"],
        "word_tokens": words,
        "word_labels": row["word_labels"],
        "tokens": tokens,
        "input_ids": input_ids,
        "token_labels": token_labels
    }
    tokenized_samples.append(tokenized_sample)

print("Example tokenized sample:")
print(tokenized_samples[0])


In [ ]:
with open(f"{PROJECT_ROOT}/v03/data/data_wordpiece_tokenized_biobert.jsonl", "w") as f:
    for sample in tokenized_samples:
        f.write(json.dumps(sample) + "\n")

In [ ]:
# Create ids2label and labels2ids mappings

In [ ]:
dataset = []
with open(f"{PROJECT_ROOT}/v03/data/data_wordpiece_tokenized_biobert.jsonl", "r") as f:
    for line in f:
        dataset.append(json.loads(line))
        

In [ ]:
unique_labels = set()

for row in dataset: 
    for lbl in row["token_labels"]:
        if lbl != "None":  # ignore special tokens
            unique_labels.add(lbl)

# Sort for stable ordering
unique_labels = sorted(list(unique_labels))
# Create mappings
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print("Number of labels:", len(label2id))

# Save mapping:
with open(f"{PROJECT_ROOT}/v03/data/label2id_biobert.json", "w") as f:
    json.dump(label2id, f, indent = 2)
with open(f"{PROJECT_ROOT}/v03/data/id2label_biobert.json", "w") as f:
    json.dump(id2label, f, indent = 2)

In [ ]:
# 05/03/2026 - MISSING ROW OF with the numeric token ids!
dataset = []
with open(f"{PROJECT_ROOT}/v03/data/label2id.json", "r") as f:
    label2id = json.load(f)
    
with open(f"{PROJECT_ROOT}/v03/data/data_wordpiece_tokenized_biobert.jsonl", "r") as f:
    for line in f:
        dataset.append(json.loads(line))


for row in dataset:
    row["token_label_ids"] = [
        -100 if lbl == "None" else label2id[lbl]
        for lbl in row["token_labels"]
    ]


In [ ]:
dataset[0]

In [ ]:
with open("data/data_wordpiece_tokenized_biobert.jsonl", "w") as f:
    for sample in dataset:
        f.write(json.dumps(sample) + "\n")